# What happens when you actually ask it something

MichAl Academy, unit 4.5.

Run each cell with **Shift+Enter**.

Unit 4.4 trained a model. Nothing so far says what happens at the moment somebody
types a question into it. Four things do, and all four are measured here: how the
next character gets chosen, how far back the model can see, what gets kept in
memory so the work is not repeated, and why the same question twice can give two
different answers.

The model is the one from unit 4.4, trained again here so this notebook stands
alone. It reads one character at a time, which keeps it small enough to train on
a laptop CPU and makes every effect below visible without a GPU.


In [ ]:
import time
import warnings

import numpy as np
import torch
from sklearn.datasets import fetch_20newsgroups
from transformers import AutoConfig
from torch import nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

CTX = 128                        # how many characters the model can see at once
D, HEADS, BLOCKS = 96, 4, 2

news = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
raw = "\n".join(news.data)
raw = "".join(c for c in raw if 32 <= ord(c) < 127 or c == "\n")[:300_000]

chars = sorted(set(raw))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
VOCAB = len(chars)
data = np.array([stoi[c] for c in raw], dtype=np.int64)
print(f"corpus {len(raw):,} characters, vocabulary {VOCAB} characters")
print(f"context window {CTX} characters")


## The model

The same stack of masked transformer blocks unit 4.4 trained, with one addition
used later in this notebook: each block can be asked to remember the keys and
values it has already computed, instead of computing them again.


In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))
        self.proj = nn.Linear(d, d)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))

    def split(self, t, B, L):
        return t.view(B, L, self.heads, t.shape[-1] // self.heads).transpose(1, 2)

    def forward(self, x, mask, cache=None):
        h = self.n1(x)
        B, L, d = h.shape
        q = self.split(self.q(h), B, L)
        k = self.split(self.k(h), B, L)
        v = self.split(self.v(h), B, L)

        if cache is not None:
            # Keys and values for every earlier position were already computed
            # on an earlier call. Reuse them and append only the new ones.
            if cache["k"] is not None:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v

        dh = d // self.heads
        s = (q @ k.transpose(-2, -1)) / dh ** 0.5
        if mask is not None:
            s = s.masked_fill(mask, float("-inf"))
        s = s.softmax(dim=-1)
        x = x + self.proj((s @ v).transpose(1, 2).reshape(B, L, d))
        return x + self.mlp(self.n2(x))


class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, D)
        self.pos = nn.Embedding(CTX, D)
        self.blocks = nn.ModuleList([Block(D, HEADS) for _ in range(BLOCKS)])
        self.norm = nn.LayerNorm(D)
        self.out = nn.Linear(D, VOCAB)
        self.register_buffer("mask",
                             torch.triu(torch.ones(CTX, CTX, dtype=torch.bool), 1))

    def forward(self, x, caches=None, offset=0):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(offset, offset + L))
        if caches is None:
            m = self.mask[:L, :L]
            for b in self.blocks:
                h = b(h, m)
        else:
            # Prefill: several positions at once, so they still have to be
            # stopped from reading each other's futures. A single new position
            # attending to everything already cached needs no mask, because
            # there is nothing after it to hide.
            m = self.mask[offset:offset + L, :offset + L] if L > 1 else None
            for b, c in zip(self.blocks, caches):
                h = b(h, m, cache=c)
        return self.out(self.norm(h))


model = CharLM()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")


In [ ]:
rng = np.random.default_rng(0)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

t0 = time.time()
for step in range(4000):
    i = rng.integers(0, len(data) - CTX - 1, 24)
    x = torch.tensor(np.stack([data[j:j + CTX] for j in i]))
    y = torch.tensor(np.stack([data[j + 1:j + CTX + 1] for j in i]))
    opt.zero_grad()
    loss = loss_fn(model(x).reshape(-1, VOCAB), y.reshape(-1))
    loss.backward()
    opt.step()
    if step % 800 == 0:
        print(f"step {step:4d}  loss {loss.item():.3f}")
model.eval()
print(f"final loss {loss.item():.3f}   ({time.time() - t0:.0f}s)")


## 1. Choosing the next character

The model does not output a character. It outputs a score for every character in
the vocabulary, and something has to turn those scores into one choice.

**Greedy** decoding takes the highest-scoring character every time. **Sampling**
draws one at random, in proportion to the scores. **Temperature** scales the
scores before drawing: below 1 it sharpens them towards the favourite, above 1 it
flattens them towards an even draw. Temperature 0 is greedy.

The two cells below measure what that setting costs and buys. Two things are
counted over generated text:

- **Looping**: the share of generated positions whose preceding 12 characters
  have already appeared earlier in the same generated text. A model that gets
  stuck repeating itself scores high.
- **Real words**: the share of generated whitespace-separated words that appear
  in the training corpus. A model producing letter soup scores low.


In [ ]:
@torch.no_grad()
def generate(model, prompt, n, temp, seed):
    g = torch.Generator().manual_seed(seed)
    ids = [stoi[c] for c in prompt]
    for _ in range(n):
        x = torch.tensor(ids[-CTX:])[None]
        logits = model(x)[0, -1]
        if temp == 0:
            nxt = int(logits.argmax())
        else:
            p = (logits / temp).softmax(dim=-1)
            nxt = int(torch.multinomial(p, 1, generator=g))
        ids.append(nxt)
    return "".join(itos[i] for i in ids)


CORPUS_WORDS = set(raw.split())


def looping(text, k=12):
    seen, hits = set(), 0
    for i in range(k, len(text)):
        w = text[i - k:i]
        if w in seen:
            hits += 1
        seen.add(w)
    return hits / max(1, len(text) - k)


def real_words(text):
    ws = text.split()
    return sum(w in CORPUS_WORDS for w in ws) / max(1, len(ws))


PROMPT = "The question is whether "
print(f"{'temperature':>12}  {'runs':>5}  {'looping':>8}  {'real words':>11}")
rows = []
for temp in (0.0, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0):
    # Temperature 0 is greedy and therefore deterministic: five runs would be
    # five copies of one run, so it gets one.
    seeds = range(1) if temp == 0 else range(5)
    loops, words = [], []
    for seed in seeds:
        g = generate(model, PROMPT, 400, temp, seed)[len(PROMPT):]
        loops.append(looping(g))
        words.append(real_words(g))
    rows.append((temp, float(np.mean(loops)), float(np.mean(words))))
    print(f"{temp:>12.1f}  {len(loops):>5}  {rows[-1][1]:>8.3f}  {rows[-1][2]:>11.3f}")


In [ ]:
for temp in (0.0, 0.8, 1.5):
    print(f"--- temperature {temp} ---")
    print(generate(model, PROMPT, 180, temp, seed=0))
    print()


## 2. How far back it can see

`CTX` is 128, and it is a hard limit rather than a preference. The position
embedding table has 128 rows, and the generation loop above slices `ids[-CTX:]`.
A character 129 places back is not weighted less. It is not there.

The cell below measures what the model does with the context it has: the loss on
predicting one character, given only the last **k** characters, for k from 1 up
to the full window. Lower is better.


In [ ]:
@torch.no_grad()
def loss_at_context(k, n=400, seed=0):
    r = np.random.default_rng(seed)
    starts = r.integers(CTX, len(data) - 1, n)
    xs = torch.tensor(np.stack([data[s - k:s] for s in starts]))
    ys = torch.tensor(np.array([data[s] for s in starts]))
    return loss_fn(model(xs)[:, -1, :], ys).item()


# One draw of 400 positions cannot say whether a small wobble is real, so every
# point is measured on five independent draws and the spread is reported with it.
print(f"{'context':>8}  {'mean loss':>10}  {'spread':>8}")
ctx_rows = []
for k in (1, 2, 4, 8, 16, 32, 64, 128):
    vals = [loss_at_context(k, seed=s) for s in range(5)]
    ctx_rows.append((k, float(np.mean(vals)), float(np.std(vals))))
    print(f"{k:>8}  {ctx_rows[-1][1]:>10.4f}  {ctx_rows[-1][2]:>8.4f}")
print()
best = min(ctx_rows, key=lambda r: r[1])
print(f"best at {best[0]} characters: {best[1]:.4f}")
print(f"full window (128): {ctx_rows[-1][1]:.4f}, "
      f"{abs(ctx_rows[-1][1] - best[1]):.4f} worse, "
      f"against a spread of about {np.mean([r[2] for r in ctx_rows]):.4f}")


## 3. The KV cache

Generating 128 characters means 128 forward passes. Without a cache each pass
recomputes the keys and values for every character before it, which is work
already done. Storing them costs memory and saves that work.

The two cells below generate the same 128 characters twice and time both. They
also check the outputs match, because a speed-up that changes the answer is not
a speed-up.


In [ ]:
@torch.no_grad()
def gen_cached(model, prompt, n):
    caches = [{"k": None, "v": None} for _ in model.blocks]
    ids = [stoi[c] for c in prompt]
    logits = model(torch.tensor(ids)[None], caches=caches, offset=0)[0, -1]
    for _ in range(n):
        ids.append(int(logits.argmax()))
        logits = model(torch.tensor([ids[-1]])[None],
                       caches=caches, offset=len(ids) - 1)[0, -1]
    return "".join(itos[i] for i in ids)


# One length cannot show a saving that is supposed to grow, so this times two.
print(f"{'characters':>10}  {'no cache':>10}  {'cache':>9}  {'ratio':>6}  {'same text':>10}")
ratios = {}
for n in (25, 100):
    t0 = time.time(); plain = generate(model, PROMPT, n, 0.0, 0); t_plain = time.time() - t0
    t0 = time.time(); cached = gen_cached(model, PROMPT, n); t_cached = time.time() - t0
    ratios[n] = t_plain / t_cached
    print(f"{n:>10}  {t_plain:>9.4f}s  {t_cached:>8.4f}s  {ratios[n]:>5.2f}x  "
          f"{str(plain == cached):>10}")

# If the uncached path were pure attention it would grow as n squared and the
# cached one as n, so the ratio itself should grow in step with n. It does not,
# and the size of the miss is worth printing rather than waving at.
predicted = ratios[25] * (100 / 25)
print()
print(f"if the saving were pure n-squared, the ratio at 100 would be about "
      f"{predicted:.1f}x; measured {ratios[100]:.2f}x")
print("the rest is fixed per-pass cost, which dominates at this model size")


The memory cost is arithmetic rather than a measurement, so here it is for this
notebook's model and for a real one. **This model's window is 128, so anything
past that is a what-if**; SmolLM2-135M is a model students can actually run, and
its numbers are its own.


In [ ]:
per_pos = 2 * BLOCKS * D * 4        # key and value, every block, float32
weights = sum(p.numel() for p in model.parameters()) * 4
print(f"this notebook's model: {per_pos:,} bytes per position "
      f"(2 x {BLOCKS} blocks x {D} numbers x 4 bytes)")
print(f"  its whole 128-position window: {per_pos * 128 / 1024:.0f} KiB")
print(f"  its weights:                   {weights / 1024**2:.1f} MiB")
print()

cfg = AutoConfig.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
kv_heads = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
head_dim = cfg.hidden_size // cfg.num_attention_heads
sm_per_pos = 2 * cfg.num_hidden_layers * kv_heads * head_dim * 2   # float16
print(f"SmolLM2-135M: {cfg.num_hidden_layers} layers, {kv_heads} key/value heads, "
      f"head dim {head_dim}, window {cfg.max_position_embeddings:,}")
print(f"  {sm_per_pos:,} bytes per token at float16")
print(f"  a full {cfg.max_position_embeddings:,}-token context: "
      f"{sm_per_pos * cfg.max_position_embeddings / 1024**2:.0f} MiB")
print(f"  its weights at float16: {134_515_008 * 2 / 1024**2:.0f} MiB")


## 4. Why the same question gives two answers

Two different things are at work, and only one of them is under your control.

**Sampling** is the obvious one: temperature above zero means drawing at random,
so the same prompt gives a different answer every time by design.

The second is the one that surprises people. At temperature zero there is no
randomness left, so the answer should be identical every time, and on a hosted
model it often is not. **The usual explanation, that floating-point arithmetic
plus parallel hardware makes results random, is wrong**, and the cells below
check it here rather than repeating it.


In [ ]:
X = torch.tensor(np.stack([data[s - CTX:s] for s in
                           np.random.default_rng(0).integers(CTX, len(data) - 1, 64)]))

with torch.no_grad():
    a = model(X[:8])
    b = model(X[:8])
    print("differences below are over the model's output scores, one per character")
    print(f"in the vocabulary, for every position:")
    print(f"same call twice, largest difference        {(a - b).abs().max():.3e}")

    alone = torch.stack([model(X[i:i + 1])[0, -1] for i in range(64)])
    for bs in (8, 16, 64):
        batched = torch.cat([model(X[i:i + bs])[:, -1, :] for i in range(0, 64, bs)])
        d = (alone - batched).abs().max().item()
        ch = (alone.argmax(-1) != batched.argmax(-1)).float().mean().item()
        print(f"alone vs batch of {bs:>2}, largest difference   {d:.3e}   "
              f"chosen character changed {ch:.3f}")


So on this machine the arithmetic is repeatable to the last bit, and batching
changes nothing. That is not because floating point is exact. It is not:


In [ ]:
torch.manual_seed(0)
v = torch.randn(10_000, dtype=torch.float32)   # standard normal, mean 0, sd 1
print(f"10,000 standard normal numbers, summed forwards and backwards, differ by "
      f"{abs(v.sum().item() - v.flip(0).sum().item()):.3e}")
print(f"their total is about {v.sum().item():.4f}, so that is a relative error of "
      f"{abs(v.sum().item() - v.flip(0).sum().item()) / abs(v.sum().item()):.2e}")
print(f"in float32, (0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3) is "
      f"{(0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3)}")


**Order of addition matters, and the order here never changed.** That is the
whole point. A kernel is deterministic given the same input *and the same
shape*; what varies on a busy server is the shape, because your request is
batched together with however many other requests happened to arrive at that
moment. Thinking Machines Lab call this batch invariance and measured the
consequence: 1,000 completions of one prompt at temperature 0 from Qwen3-235B
gave **80 different answers**, identical for the first 102 tokens and diverging
at token 103. With batch-invariant kernels substituted, all 1,000 were identical.

Nothing in this notebook can reproduce that, because there is no server and no
varying batch. What it can show is the part that is under your control:


In [ ]:
g1 = generate(model, PROMPT, 60, 0.0, seed=0)
g2 = generate(model, PROMPT, 60, 0.0, seed=99)
s1 = generate(model, PROMPT, 60, 1.0, seed=0)
s2 = generate(model, PROMPT, 60, 1.0, seed=99)
print(f"greedy twice, different seeds:  identical = {g1 == g2}")
print(f"sampled twice, different seeds: identical = {s1 == s2}")


## What this unit measured

- **Temperature is a trade with no free end.** Every step towards coherent text
  is a step towards repeating itself, and every step away from repetition is a
  step towards letter soup. The table in section 1 is that trade, priced.
- **The context window is a wall, not a slope.** Section 2 shows the loss falling
  as context grows and then flattening. Past `CTX` there is nothing to measure,
  because the characters are not passed in.
- **The KV cache changes speed and nothing else**, which section 3 checks rather
  than assumes.
- **The popular explanation for non-determinism is wrong**, and section 4 checks
  it rather than repeating it: the arithmetic here is repeatable to the last bit
  and batching changes nothing. Order of addition does matter, and on a server
  the order changes because the batch you are in changes.
